# NB35 — Why doesn't merA track Hg? Three computational tests

**Hypothesis:** merA (K00520) distribution reflects historical Hg selection in felsic/hydrothermal settings via HGT, not contemporary soil Hg concentrations.

- **Test 1:** Fritz & Purvis D predicts cognate-metal non-specificity
- **Test 2:** Metal PCA — merA tracks a felsic factor distinct from the Hg factor
- **Test 3:** Legacy mine proximity predicts merA CWM beyond current USGS Hg

**Prerequisites:** NB33, NB34, nb02_otu_long_cache.parquet

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys, warnings, urllib.request
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H, grid_h
apply_style()

ROOT   = Path('/home/hmacgregor/BERIL-research-observatory')
CME    = ROOT / 'projects/comprehensive_metal_ecology'
USAENV = ROOT / 'projects/usa_env_bioindicators'
DATA   = CME / 'data'
FIGS   = CME / 'figures'

for p in [DATA / 'nb33_sample_master.parquet', DATA / 'nb33_mwas_results.csv',
          DATA / 'fritz_purvis_D_genome.csv',   DATA / 'curated_mrg_ko_ids_v2.csv',
          DATA / 'nb34_genus_mwas_ko_annotated.csv',
          USAENV / 'data/nb02_otu_long_cache.parquet']:
    assert p.exists(), f'Missing: {p}'
print('Setup done.')


Setup done.


## Test 1 — Fritz & Purvis D predicts cognate-metal non-specificity

In [2]:
d_df    = pd.read_csv(DATA / 'fritz_purvis_D_genome.csv')
mwas    = pd.read_csv(DATA / 'nb33_mwas_results.csv')
curated = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')

KS = 'pH+SOC+WTD+redox+MAT'
mwas_ks = mwas[mwas['ctrl'] == KS].copy()
all_metals_tested = set(mwas_ks['metal'].unique())
print(f'Kitchen-sink MWAS: {len(mwas_ks)} rows, {len(all_metals_tested)} metals, '
      f'{mwas_ks["ko"].nunique()} KOs')

METAL_MAP = {
    'mercury': 'hg', 'arsenic': 'as', 'chromium': 'cr', 'copper': 'cu',
    'nickel':  'ni', 'cobalt': 'co',  'zinc': 'zn',     'cadmium': 'cd',
    'silver':  'ag', 'lead':   'pb',  'sulfur': 's',     'iron': 'fe',
}

def parse_metals(s):
    if pd.isna(s) or str(s).strip() == '':
        return []
    out = []
    for m in str(s).split(','):
        m = m.strip().lower()
        m = METAL_MAP.get(m, m)
        if m in all_metals_tested:
            out.append(m)
    return out

curated['cognate_norm'] = curated['metals'].apply(parse_metals)
curated_with_metals = curated[curated['cognate_norm'].map(len) > 0]
print(f'KOs with mapped cognate metals: {curated_with_metals["KO"].nunique()}')


Kitchen-sink MWAS: 7240 rows, 48 metals, 151 KOs
KOs with mapped cognate metals: 286


In [3]:
records = []
for _, d_row in d_df.iterrows():
    ko = d_row['ko_id']
    D  = d_row['D']

    cog_rows = curated_with_metals[curated_with_metals['KO'] == ko]
    if cog_rows.empty:
        continue
    cognate = set()
    for _, cr in cog_rows.iterrows():
        cognate.update(cr['cognate_norm'])
    if not cognate:
        continue

    ko_mwas = mwas_ks[mwas_ks['ko'] == ko].copy()
    if ko_mwas.empty:
        continue
    ko_mwas = ko_mwas.sort_values('pval_fdr').reset_index(drop=True)
    metal_to_rank = {r['metal']: i + 1 for i, r in ko_mwas.iterrows()}
    n_metals = len(metal_to_rank)

    cognate_ranks = [metal_to_rank[m] for m in cognate if m in metal_to_rank]
    if not cognate_ranks:
        continue

    records.append({
        'ko_id':             ko,
        'gene_name':         d_row['gene_name'],
        'subcategory':       d_row['subcategory'],
        'D':                 D,
        'cognate_metals':    ','.join(sorted(cognate)),
        'best_cognate_rank': min(cognate_ranks),
        'n_metals':          n_metals,
        'rank_pct':          min(cognate_ranks) / n_metals,
        'p_conserved':       d_row['p_conserved'],
        'p_random':          d_row['p_random'],
    })

rank_df = pd.DataFrame(records)
print(f'KOs matched (D + cognate rank): {len(rank_df)}')

rho_t1, pval_t1 = stats.spearmanr(rank_df['D'], rank_df['rank_pct'])
print(f'Spearman rho(D, cognate_rank_pct) = {rho_t1:.3f}, p = {pval_t1:.4f}, N = {len(rank_df)}')

mera = rank_df[rank_df['ko_id'] == 'K00520']
if not mera.empty:
    print(f'merA: D={mera["D"].values[0]:.3f}, '
          f'rank={mera["best_cognate_rank"].values[0]}/{mera["n_metals"].values[0]} '
          f'(pct={mera["rank_pct"].values[0]:.3f})')

print('\nTop 10 by rank_pct (worst cognate specificity):')
print(rank_df.sort_values('rank_pct', ascending=False)
      [['gene_name', 'subcategory', 'D', 'cognate_metals', 'best_cognate_rank', 'rank_pct']]
      .head(10).to_string())


KOs matched (D + cognate rank): 53
Spearman rho(D, cognate_rank_pct) = 0.092, p = 0.5124, N = 53
merA: D=0.646, rank=29/48 (pct=0.604)

Top 10 by rank_pct (worst cognate specificity):
   gene_name                subcategory         D cognate_metals  best_cognate_rank  rank_pct
7      STAR2      Transport/Homeostasis  0.187171             ni                 44  0.916667
21      nikR         Sensing/Regulation  0.127686             ni                 42  0.875000
41      merE      Transport/Homeostasis  0.727684             hg                 41  0.854167
35      cueO  Resistance/Detoxification -0.275060             cu                 40  0.833333
42      golS         Sensing/Regulation  0.264594             au                 40  0.833333
22      nrsD  Resistance/Detoxification  0.821262             ni                 39  0.812500
30      troC      Transport/Homeostasis  0.102730             zn                 34  0.708333
25      aoxA  Resistance/Detoxification  0.423432             as

In [4]:
fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))
grid_h(ax)

cats = sorted(rank_df['subcategory'].dropna().unique())
cat_colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(cats)}

for cat, grp in rank_df.groupby('subcategory'):
    ax.scatter(grp['D'], grp['rank_pct'],
               color=cat_colors[cat], label=cat, alpha=0.75, s=45,
               edgecolor='k', linewidth=0.4)

if not mera.empty:
    ax.scatter(mera['D'], mera['rank_pct'],
               color='red', s=100, edgecolor='k', linewidth=0.8, zorder=6)
    ax.annotate('merA',
                xy=(mera['D'].values[0], mera['rank_pct'].values[0]),
                xytext=(8, -12), textcoords='offset points', fontsize=8,
                color='red', fontweight='bold')

x, y = rank_df['D'].values, rank_df['rank_pct'].values
coefs = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, np.polyval(coefs, x_line), color='gray', lw=0.9, ls='--')

ax.set_xlabel('Fritz & Purvis D  (higher = more mobile via HGT)')
ax.set_ylabel('Cognate-metal rank (fraction; 0 = top hit, 1 = bottom)')
ax.set_title(
    f'High HGT signal → poorer cognate-metal tracking\n'
    f'Spearman rho = {rho_t1:.2f}, p = {pval_t1:.3f}, N = {len(rank_df)}',
    fontsize=10)
ax.legend(fontsize=7, loc='upper left', framealpha=0.8)

save(fig, FIGS / 'fig_nb35_d_vs_cognate_rank')
print('Saved fig_nb35_d_vs_cognate_rank.pdf')


Saved fig_nb35_d_vs_cognate_rank.pdf


## Test 2 — Metal PCA: does merA track the Hg factor or the felsic/hydrothermal factor?

In [5]:
samp = pd.read_parquet(DATA / 'nb33_sample_master.parquet')

METAL_COLS = sorted([c for c in samp.columns
                     if c.startswith('usgs_')
                     and samp[c].dtype in [np.float64, np.float32, float]])
print(f'Metal columns: {len(METAL_COLS)}')

metals_log = np.log1p(samp[METAL_COLS].copy())
metals_log.columns = [c.replace('usgs_', '') for c in METAL_COLS]

# Drop columns with >50% NA (Re, Pd, Pt, Ta, Tb, Sm, Lu, Hf, Ag, Ge have 70-100% NA)
na_frac = metals_log.isna().mean()
keep_cols = na_frac[na_frac < 0.5].index.tolist()
metals_log = metals_log[keep_cols]
print(f'Columns after <50% NA filter: {len(keep_cols)} (dropped {len(METAL_COLS) - len(keep_cols)} high-NA metals)')

metals_complete = metals_log.dropna()
print(f'Complete-case samples: {len(metals_complete)}')

scaler = StandardScaler()
X = scaler.fit_transform(metals_complete)

pca = PCA(n_components=min(10, X.shape[1]))
pca.fit(X)
loadings = pd.DataFrame(
    pca.components_.T,
    index=metals_complete.columns,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
)
pct = pca.explained_variance_ratio_ * 100
print('Variance explained:', np.round(pct[:6], 1))

# merA-tracked metals: significant at kitchen-sink MWAS
mera_mwas_ks = mwas_ks[mwas_ks['ko'] == 'K00520'].sort_values('pval_fdr')
mera_sig = mera_mwas_ks[mera_mwas_ks['sig_fdr']]['metal'].tolist()
print(f'merA significant metals at kitchen-sink (n={len(mera_sig)}): {mera_sig}')

if 'hg' in loadings.index:
    print('Hg PC loadings (PC1-6):')
    print(loadings.loc['hg', [f'PC{i+1}' for i in range(6)]].round(3).to_string())
if mera_sig:
    avail = [m for m in mera_sig if m in loadings.index]
    print(f'merA-tracked metals mean PC loadings (PC1-6, n={len(avail)}):')
    print(loadings.loc[avail, [f'PC{i+1}' for i in range(6)]].mean().round(3).to_string())


Metal columns: 50
Columns after <50% NA filter: 32 (dropped 18 high-NA metals)
Complete-case samples: 4554
Variance explained: [56.2 20.3 15.3  4.   1.9  1.2]
merA significant metals at kitchen-sink (n=22): ['nd', 'sb', 'ce', 'nb', 'ba', 'se', 'yb', 'y', 'as', 'sn', 'la', 'tl', 'sr', 'u', 'pb', 'zr', 'mo', 'bi', 'th', 'w', 'v', 'pd']
Hg PC loadings (PC1-6):
PC1    0.003
PC2    0.365
PC3    0.023
PC4   -0.243
PC5   -0.081
PC6    0.298
merA-tracked metals mean PC loadings (PC1-6, n=18):
PC1    0.194
PC2   -0.010
PC3   -0.025
PC4   -0.008
PC5    0.030
PC6   -0.020


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H),
                         gridspec_kw={'wspace': 0.35})

ax = axes[0]
grid_h(ax)

def metal_color(m):
    if m == 'hg':       return 'crimson'
    if m in mera_sig:   return PALETTE[0]
    return '#cccccc'

colors = [metal_color(m) for m in loadings.index]
ax.scatter(loadings['PC1'], loadings['PC2'],
           c=colors, s=55, edgecolor='k', linewidth=0.4, alpha=0.9)

label_set = {'hg'} | set(mera_sig[:8])
for m, row in loadings.iterrows():
    if m in label_set:
        ax.annotate(m.upper(), (row['PC1'], row['PC2']),
                    fontsize=6.5, xytext=(3, 3), textcoords='offset points')

legend_els = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='crimson',  markersize=7, label='Hg'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=PALETTE[0], markersize=7, label='merA-tracked'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#cccccc',  markersize=7, label='Other'),
]
ax.legend(handles=legend_els, fontsize=7)
ax.set_xlabel(f'PC1 ({pct[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({pct[1]:.1f}%)')
ax.set_title('Metal PCA loadings', fontsize=10)

ax = axes[1]
grid_h(ax)
n_pcs = 6
pc_names = [f'PC{i+1}' for i in range(n_pcs)]

hg_loads   = loadings.loc['hg', pc_names].values if 'hg' in loadings.index else np.zeros(n_pcs)
avail      = [m for m in mera_sig if m in loadings.index]
mera_loads = loadings.loc[avail, pc_names].mean().values if avail else np.zeros(n_pcs)

x = np.arange(n_pcs)
w = 0.35
ax.bar(x - w/2, hg_loads,   width=w, label='Hg',          color='crimson',  edgecolor='k', linewidth=0.5)
ax.bar(x + w/2, mera_loads, width=w, label='merA-tracked', color=PALETTE[0], edgecolor='k', linewidth=0.5)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(x);  ax.set_xticklabels(pc_names)
ax.set_xlabel('Principal component')
ax.set_ylabel('Mean loading')
ax.set_title('Hg vs merA-tracked metals', fontsize=10)
ax.legend(fontsize=7)

fig.suptitle('Test 2: merA tracks a different geochemical factor than Hg', y=1.02)
save(fig, FIGS / 'fig_nb35_metal_pca')
print('Saved fig_nb35_metal_pca.pdf')


Saved fig_nb35_metal_pca.pdf


## Test 3 — Legacy Hg mine proximity predicts merA CWM beyond current USGS Hg

In [7]:
# merA presence fraction per genus from NB34 output
nb34_ann = pd.read_csv(DATA / 'nb34_genus_mwas_ko_annotated.csv')
mera_pres = (nb34_ann[nb34_ann['ko'] == 'K00520'][['genus', 'pres_frac', 'n_genomes']]
             .drop_duplicates('genus')
             .rename(columns={'genus': 'genus_lower'}))
print(f'merA pres_frac: {len(mera_pres)} genera')
print(f'  max={mera_pres["pres_frac"].max():.2f}, '
      f'median={mera_pres["pres_frac"].median():.3f}')

# Relative abundance within each sample
otu = pd.read_parquet(USAENV / 'data/nb02_otu_long_cache.parquet')
otu['relabund'] = otu.groupby('sample_id')['count'].transform(lambda x: x / x.sum())

# CWM(merA) = sum(relabund × pres_frac) per sample
otu_mera = otu.merge(mera_pres[['genus_lower', 'pres_frac']], on='genus_lower', how='inner')
cwm = (otu_mera.groupby('sample_id')
       .apply(lambda g: (g['relabund'] * g['pres_frac']).sum(), include_groups=False)
       .reset_index(name='mera_cwm'))
sample_coords = otu.groupby('sample_id')[['lat', 'lon']].first().reset_index()
cwm = cwm.merge(sample_coords, on='sample_id', how='left')

print(f'merA CWM computed for {len(cwm)} samples')
print(f'CWM: mean={cwm["mera_cwm"].mean():.5f}, '
      f'std={cwm["mera_cwm"].std():.5f}, '
      f'max={cwm["mera_cwm"].max():.4f}')


merA pres_frac: 68 genera
  max=0.50, median=0.064


merA CWM computed for 6014 samples
CWM: mean=0.02510, std=0.02643, max=0.4033


In [8]:
# Load MRDS mercury mine locations
# Source: https://mrdata.usgs.gov/mrds/mrds-csv.zip (304,632 mineral deposit records)
# Filtered to continental US mercury sites (commod1/2/3 == 'mercury'), with valid lat/lon
MRDS_PATH = DATA / 'mrds_hg_continental_us.csv'

if MRDS_PATH.exists():
    mines = pd.read_csv(MRDS_PATH)
    mines['lat'] = pd.to_numeric(mines['lat'], errors='coerce')
    mines['lon'] = pd.to_numeric(mines['lon'], errors='coerce')
    mines = mines.dropna(subset=['lat', 'lon'])
    print(f'MRDS full: {len(mines)} continental US mercury sites')
    print('Top states:', mines['state'].value_counts().head(6).to_dict())
    prod = mines[mines['dev_stat'].isin(['Past Producer', 'Producer'])]
    print(f'Known producers only: {len(prod)} sites')
else:
    # Hardcoded major districts as fallback
    mines = pd.DataFrame([
        {'site_name': 'New Idria CA',    'lat': 36.23, 'lon': -120.67},
        {'site_name': 'New Almaden CA',  'lat': 37.18, 'lon': -121.82},
        {'site_name': 'Sulphur Bank CA', 'lat': 39.00, 'lon': -122.90},
        {'site_name': 'Terlingua TX',    'lat': 29.32, 'lon': -103.60},
        {'site_name': 'McDermitt NV',    'lat': 41.92, 'lon': -117.65},
        {'site_name': 'Opalite OR',      'lat': 42.30, 'lon': -118.70},
        {'site_name': 'Marysvale UT',    'lat': 38.45, 'lon': -112.23},
    ])
    print(f'Using hardcoded fallback: {len(mines)} major Hg districts')

print(mines[['site_name', 'state', 'lat', 'lon']].head(5).to_string()
      if 'site_name' in mines.columns else mines.head(5).to_string())


MRDS full: 2651 continental US mercury sites
Top states: {'California': 1002, 'Nevada': 691, 'Oregon': 616, 'Arkansas': 68, 'Washington': 65, 'Arizona': 65}
Known producers only: 1020 sites
          site_name       state       lat        lon
0        Loop Ranch  California  35.22559 -118.53812
1              Mine  California  38.77377 -122.71746
2      Marcus Stein  Washington  48.19676 -120.74397
3  Adobe Walls Mine       Texas  29.47333 -103.55222
4  Mercury Prospect        Utah  38.15274 -113.53998


In [9]:
from scipy.spatial import cKDTree
from statsmodels.formula.api import ols as sm_ols

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2)
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

mine_xy = mines[['lat', 'lon']].dropna().values
samp_xy = cwm[['lat', 'lon']].values

tree = cKDTree(np.radians(mine_xy))
_, idx = tree.query(np.radians(samp_xy), k=1)
dist_km = np.array([
    haversine_km(samp_xy[i, 0], samp_xy[i, 1], mine_xy[idx[i], 0], mine_xy[idx[i], 1])
    for i in range(len(samp_xy))
])
cwm['dist_to_mine_km'] = dist_km
print(f'Dist to nearest mine: min={dist_km.min():.1f} km, '
      f'median={np.median(dist_km):.1f} km, max={dist_km.max():.1f} km')

cov_cols = ['sample_id', 'usgs_hg', 'soil_ph', 'soil_soc', 'wtd_m', 'p_oxic', 'wc_mat']
samp_cov = samp[cov_cols].copy()
anal = cwm.merge(samp_cov, on='sample_id', how='inner')
anal = anal.dropna(subset=['mera_cwm', 'dist_to_mine_km', 'usgs_hg',
                            'soil_ph', 'soil_soc', 'wtd_m', 'p_oxic', 'wc_mat'])
print(f'Analysis N (complete cases): {len(anal)}')

anal['log_dist'] = np.log1p(anal['dist_to_mine_km'])
anal['log_hg']   = np.log1p(anal['usgs_hg'])
anal['log_mera'] = np.log1p(anal['mera_cwm'])

m_full    = sm_ols('log_mera ~ log_dist + log_hg + soil_ph + soil_soc + wtd_m + p_oxic + wc_mat',
                    data=anal).fit()
m_no_dist = sm_ols('log_mera ~            log_hg + soil_ph + soil_soc + wtd_m + p_oxic + wc_mat',
                    data=anal).fit()
m_no_hg   = sm_ols('log_mera ~ log_dist +          soil_ph + soil_soc + wtd_m + p_oxic + wc_mat',
                    data=anal).fit()

pr2_dist = m_full.rsquared - m_no_dist.rsquared
pr2_hg   = m_full.rsquared - m_no_hg.rsquared
b_dist, p_dist = m_full.params['log_dist'], m_full.pvalues['log_dist']
b_hg,   p_hg   = m_full.params['log_hg'],   m_full.pvalues['log_hg']

print(f'Full model R2 = {m_full.rsquared:.4f}')
print(f'Partial R2(log_dist_to_mine) = {pr2_dist:.5f}  b={b_dist:.4f}  p={p_dist:.3e}')
print(f'Partial R2(log_USGS_Hg)      = {pr2_hg:.5f}  b={b_hg:.4f}  p={p_hg:.3e}')

rho_dist, rp_dist = stats.spearmanr(anal['log_dist'], anal['log_mera'])
rho_hg,   rp_hg   = stats.spearmanr(anal['log_hg'],   anal['log_mera'])
print(f'Spearman: rho(dist)={rho_dist:.3f} p={rp_dist:.3e}  |  '
      f'rho(hg)={rho_hg:.3f} p={rp_hg:.3e}')


Dist to nearest mine: min=0.6 km, median=322.5 km, max=1070.8 km
Analysis N (complete cases): 5990
Full model R2 = 0.0377
Partial R2(log_dist_to_mine) = 0.00094  b=0.0018  p=1.539e-02
Partial R2(log_USGS_Hg)      = 0.00079  b=0.0049  p=2.711e-02
Spearman: rho(dist)=0.001 p=9.414e-01  |  rho(hg)=-0.061 p=2.597e-06


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(FIGW['full'], ROW_H),
                         gridspec_kw={'wspace': 0.35})

# Panel A: CWM vs mine distance
ax = axes[0]; grid_h(ax)
ax.scatter(anal['log_dist'], anal['log_mera'],
           alpha=0.25, s=7, color=PALETTE[0], edgecolor='none')
xv = np.linspace(anal['log_dist'].min(), anal['log_dist'].max(), 200)
ax.plot(xv, m_no_hg.params['Intercept'] + m_no_hg.params['log_dist'] * xv,
        color='k', lw=1.2)
ax.set_xlabel('log(km to nearest Hg mine + 1)')
ax.set_ylabel('log(merA CWM + 1)')
ax.set_title(f'Mine proximity\nrho={rho_dist:.2f}, p={rp_dist:.2e}', fontsize=10)

# Panel B: CWM vs current USGS Hg
ax = axes[1]; grid_h(ax)
ax.scatter(anal['log_hg'], anal['log_mera'],
           alpha=0.25, s=7, color=PALETTE[1], edgecolor='none')
xv2 = np.linspace(anal['log_hg'].min(), anal['log_hg'].max(), 200)
ax.plot(xv2, m_no_dist.params['Intercept'] + m_no_dist.params['log_hg'] * xv2,
        color='k', lw=1.2)
ax.set_xlabel('log(USGS Hg ppm + 1)')
ax.set_ylabel('log(merA CWM + 1)')
ax.set_title(f'Current soil Hg\nrho={rho_hg:.2f}, p={rp_hg:.2e}', fontsize=10)

# Panel C: Partial R²
ax = axes[2]; grid_h(ax)
labels = ['Mine\nproximity', 'Current\nUSGS Hg']
vals   = [pr2_dist, pr2_hg]
clrs   = [PALETTE[0], PALETTE[1]]
bars = ax.bar(labels, vals, color=clrs, edgecolor='k', linewidth=0.5)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_ylabel('Partial R² (unique variance)')
ax.set_title('After kitchen-sink controls', fontsize=10)
for bar, val in zip(bars, vals):
    ypos = val + abs(max(vals)) * 0.02 if val >= 0 else val - abs(max(vals)) * 0.05
    ax.text(bar.get_x() + bar.get_width() / 2, ypos,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8, color='#808080')

fig.suptitle('Test 3: Legacy mine proximity vs current Hg as predictors of merA CWM', y=1.02)
save(fig, FIGS / 'fig_nb35_legacy_proximity')
print('Saved fig_nb35_legacy_proximity.pdf')


Saved fig_nb35_legacy_proximity.pdf


## Summary

In [11]:
sep = 0.0
if 'hg' in loadings.index and mera_sig:
    avail = [m for m in mera_sig if m in loadings.index]
    hg_pc1  = loadings.loc['hg', 'PC1']
    hg_pc2  = loadings.loc['hg', 'PC2']
    m_pc1   = loadings.loc[avail, 'PC1'].mean() if avail else 0.0
    m_pc2   = loadings.loc[avail, 'PC2'].mean() if avail else 0.0
    sep     = np.sqrt((hg_pc1 - m_pc1)**2 + (hg_pc2 - m_pc2)**2)

support_t1 = rho_t1 > 0 and pval_t1 < 0.05
support_t3 = (p_dist < 0.05 and b_dist < 0)

print('=' * 70)
print("NB35 SUMMARY: Why doesn't merA track Hg at community scale?")
print('=' * 70)
print()
print('TEST 1 — D vs cognate-metal rank:')
print(f'  Spearman rho = {rho_t1:.3f}, p = {pval_t1:.4f}, N = {len(rank_df)}')
print(f'  => {"SUPPORTED" if support_t1 else "NOT SIGNIFICANT"}: '
      f'{"High D predicts poor cognate tracking" if support_t1 else "No significant relationship"}')
print()
print('TEST 2 — Metal PCA:')
if 'hg' in loadings.index and mera_sig:
    avail = [m for m in mera_sig if m in loadings.index]
    print(f'  Hg: PC1={loadings.loc["hg","PC1"]:.3f}, PC2={loadings.loc["hg","PC2"]:.3f}')
    print(f'  merA-suite mean: PC1={m_pc1:.3f}, PC2={m_pc2:.3f}')
    print(f'  Euclidean separation (PC1-PC2): {sep:.3f}')
    print(f'  => {"SUPPORTED" if sep > 0.2 else "AMBIGUOUS"}: '
          f'merA-tracked suite {"occupies different factor from Hg" if sep > 0.2 else "partially overlaps Hg"}')
print()
print('TEST 3 — Legacy mine proximity:')
print(f'  b(log_dist) = {b_dist:.4f}, p = {p_dist:.3e}, partial R2 = {pr2_dist:.5f}')
print(f'  b(log_hg)   = {b_hg:.4f}, p = {p_hg:.3e}, partial R2 = {pr2_hg:.5f}')
if support_t3:
    print('  => SUPPORTED: Closer to mine => higher merA CWM (after controlling for current Hg)')
elif p_dist < 0.05:
    print('  => INVERTED: Farther from mine => higher merA CWM (unexpected direction)')
else:
    print('  => NOT SIGNIFICANT: Mine proximity does not predict merA beyond controls')


NB35 SUMMARY: Why doesn't merA track Hg at community scale?

TEST 1 — D vs cognate-metal rank:
  Spearman rho = 0.092, p = 0.5124, N = 53
  => NOT SIGNIFICANT: No significant relationship

TEST 2 — Metal PCA:
  Hg: PC1=0.003, PC2=0.365
  merA-suite mean: PC1=0.194, PC2=-0.010
  Euclidean separation (PC1-PC2): 0.420
  => SUPPORTED: merA-tracked suite occupies different factor from Hg

TEST 3 — Legacy mine proximity:
  b(log_dist) = 0.0018, p = 1.539e-02, partial R2 = 0.00094
  b(log_hg)   = 0.0049, p = 2.711e-02, partial R2 = 0.00079
  => INVERTED: Farther from mine => higher merA CWM (unexpected direction)
